# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [36]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [37]:
# TODO

df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Total units: {total_units}")
print(f"Total revenue: ${total_revenue:,.2f}; Total units: {total_units}")

Total revenue: $8,520.00
Total units: 783
Total revenue: $8,520.00; Total units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [38]:
# TODO

category_revenue = df.groupby('category')['revenue'].sum().sort_values(ascending=False)

category_table = category_revenue.to_frame(name='revenue')
category_table['share_of_total'] = (category_table['revenue'] / df['revenue'].sum()) * 100

category_table

,revenue,share_of_total
category,,
Food,4293.0,50.387324
Merch,1771.5,20.792254
Drink,1554.0,18.239437
RainGear,901.5,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [39]:
# TODO

vendor_summary = df.groupby('vendor_id').agg(
    average_order_revenue=('revenue', 'mean'),
    order_count=('revenue', 'count')
).sort_values('average_order_revenue', ascending=False)
vendor_summary['average_order_revenue'] = vendor_summary['average_order_revenue'].round(2)
print(vendor_summary)
print(f"Highest average: {vendor_summary.index[0]}")

           average_order_revenue  order_count
vendor_id                                    
V-01                       22.60           94
V-18                       21.75          108
V-05                       20.58           93
V-10                       20.31          105
Highest average: V-01


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [40]:
# TODO

merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()
merch_share = (merch_revenue / df['revenue'].sum()) * 100
print(f"Merch share of revenue: {merch_share:.1f}%")

Merch share of revenue: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [41]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor

df_merged = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)
print(f"Rows unchanged: {len(df) == len(df_merged)}")
print(f"Revenue unchanged: {df['revenue'].sum() == df_merged['revenue'].sum()}")
unmatched_vendor = df_merged.loc[df_merged['vendor_name'].isna(), 'vendor_id'].unique()
print(f"Unmatched vendor: {unmatched_vendor}")
df_merged['vendor_name'] = df_merged['vendor_name'].fillna('Unknown Vendor')

Rows unchanged: True
Revenue unchanged: True
Unmatched vendor: ['V-18']


**The unmatched vendor, and what I did about it:** V-18 was unmatched, so I labeled it as "Unknown Vendor" to keep those orders in the report.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [42]:
# TODO

pivot = pd.pivot_table(
    df_merged,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)
pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown Vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [43]:
by_category = category_table
joined = df_merged
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a. For the next game, I would suggest stocking more Food since it brought in 4,293 dollars, which was about 50.4% of all revenue. RainGear only made 901.50 dollars, so it probably does not need as much inventory. I would also look more closely at V-01 since its orders averaged 22.60 dollars, the highest of any vendor, to see what they may be doing differently.

b. I would trust Q5 the least because V-18 was not included in the vendor lookup, even though it had 108 orders. I kept those orders under “Unknown Vendor,” so the revenue is still counted, but we do not actually know which vendor that data belongs to.